# ⚠️ ARCHIVE — Extraction Oficina del Peregrino

## Statut
Ce notebook est conservé à des fins de **traçabilité méthodologique uniquement**.  
Il ne fait pas partie du pipeline actif du projet.

## Ce qui a été réalisé
- Extraction des données Oficina del Peregrino **2018** : **fonctionnelle** → 71 observations intégrées dans `data/external/oficina/2018_raw.json`
- Contrôles qualité automatiques disponibles dans `data/external/oficina/rapport.txt`

## Ce qui a été abandonné
L'extraction des données **2019 et ultérieures** a été tentée et abandonnée.

**Raison** : les données 2019+ sont publiées exclusivement via un dashboard **Power BI embedded** sur oficinadelperegrino.com. Ce format n'est pas extractible programmatiquement sans accord institutionnel préalable avec l'archidiocèse de Santiago.

## Décision projet
La source Oficina 2018 reste intégrée comme point de référence historique dans `data/processed/dataset_compostelle_2018_2024.xlsx`.  
L'acquisition des données 2019+ est classée en **sources différées** — voir `docs/SOURCES.md` pour le guide d'acquisition si un accord institutionnel est obtenu ultérieurement.

---
*Archivé au cours de la Phase 1bis*

In [1]:
"""
===============================================================================
EXTRACTEUR OFICINA DEL PEREGRINO - Stats mensuelles 2018-2024
===============================================================================
Projet   : Analyse flux pèlerins Compostelle - correction biais SJPP
Source   : https://oficinadelperegrino.com/wp-admin/admin-ajax.php
Auteur   : Florian (pipeline conçu avec Claude, session 2026-04-17)
Licence  : Usage recherche non-commerciale

CONTEXTE MÉTHODOLOGIQUE
-----------------------
Ce script récupère les statistiques mensuelles d'arrivées à Saint-Jacques-de-
Compostelle depuis l'API admin-ajax du site officiel. En demandant mes="" (vide),
l'API retourne les 12 mois d'une année en un seul appel, donc 7 requêtes
suffisent pour couvrir 2018-2024.

PRINCIPE KISS
-------------
- Une seule fonction par responsabilité (fetch / parse / save)
- Pas de dépendances exotiques : requests + json + csv standard
- Politesse serveur : pause 2s entre requêtes (l'Oficina est une petite structure)
- Logging explicite pour audit du processus

USAGE
-----
    python extract_oficina.py

SORTIES
-------
  ./oficina_raw/<annee>_raw.json    : réponse brute par année (traçabilité)
  ./oficina_raw/oficina_long.csv    : dataset consolidé format long
  ./oficina_raw/rapport.txt         : rapport de contrôle
"""
import requests
import json
import csv
import time
from pathlib import Path
from datetime import datetime

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_URL = "https://oficinadelperegrino.com/wp-admin/admin-ajax.php"
REFERER = "https://oficinadelperegrino.com/estadisticas/"
YEARS = range(2018, 2025)  # 2018 inclus, 2025 exclu → 2018-2024
PAUSE_SECONDS = 2.0        # Politesse serveur
TIMEOUT = 30

OUTPUT_DIR = Path("./oficina_raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "es-ES,es;q=0.9,fr;q=0.8",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://oficinadelperegrino.com",
    "Referer": REFERER,
}

# =============================================================================
# FONCTIONS
# =============================================================================
def fetch_year(year: int) -> list[dict]:
    """
    Récupère les stats d'une année complète via l'API admin-ajax.
    Retourne une liste de dictionnaires (un par mois).
    Raise requests.HTTPError si statut != 2xx.
    """
    payload = {
        "action": "compostelana_get_estadisticas",
        "anno": str(year),
        "mes": "",
    }
    response = requests.post(BASE_URL, data=payload, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()


def save_raw(data: list[dict], year: int) -> Path:
    """Sauvegarde la réponse brute JSON pour traçabilité."""
    path = OUTPUT_DIR / f"{year}_raw.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return path


def consolidate_to_long(all_data: dict[int, list[dict]]) -> tuple[list[str], list[dict]]:
    """
    Transforme le dict {année: [mois1_data, mois2_data, ...]} en format long.
    Détecte dynamiquement toutes les colonnes présentes (robuste aux variations
    de schéma entre années).
    """
    # Découverte dynamique du schéma : union de toutes les clés observées
    all_keys = set()
    for year_records in all_data.values():
        for record in year_records:
            all_keys.update(record.keys())
    
    # Ordre stable : clés de base d'abord, puis le reste alphabétique
    base_keys = ["anio", "mes", "mes_titulo", "nb_registros"]
    other_keys = sorted(k for k in all_keys if k not in base_keys)
    columns = base_keys + other_keys
    
    # Aplatissement en liste de dicts
    rows = []
    for year, records in all_data.items():
        for record in records:
            row = {k: record.get(k, "") for k in columns}
            rows.append(row)
    
    return columns, rows


def save_long_csv(columns: list[str], rows: list[dict]) -> Path:
    """Sauvegarde le dataset en CSV UTF-8 avec BOM (compatibilité Excel)."""
    path = OUTPUT_DIR / "oficina_long.csv"
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, delimiter=";")
        writer.writeheader()
        writer.writerows(rows)
    return path


def generate_report(all_data: dict[int, list[dict]], columns: list[str]) -> Path:
    """Génère un rapport texte de contrôle qualité."""
    path = OUTPUT_DIR / "rapport.txt"
    lines = []
    lines.append("=" * 70)
    lines.append("RAPPORT D'EXTRACTION - OFICINA DEL PEREGRINO")
    lines.append("=" * 70)
    lines.append(f"Date extraction : {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"Source          : {BASE_URL}")
    lines.append(f"Période         : {min(all_data.keys())}-{max(all_data.keys())}")
    lines.append(f"Nombre de mois  : {sum(len(v) for v in all_data.values())}")
    lines.append(f"Schéma détecté  : {len(columns)} colonnes")
    lines.append("")
    lines.append("COLONNES DÉCOUVERTES :")
    for col in columns:
        lines.append(f"  - {col}")
    lines.append("")
    lines.append("TOTAUX ANNUELS (contrôle cohérence vs AFCC) :")
    lines.append("-" * 70)
    # Références AFCC connues (extraites des notes de conjoncture)
    afcc_ref = {
        2018: 327378, 2019: 347578, 2020: 54144,
        2021: 178912, 2022: 438307, 2023: 445397,
    }
    for year in sorted(all_data.keys()):
        records = all_data[year]
        total = sum(int(r.get("nb_registros", 0)) for r in records)
        ref = afcc_ref.get(year)
        if ref:
            diff = total - ref
            status = "✓" if abs(diff) <= 5 else "⚠"
            lines.append(f"  {year} : {total:>8,}  (réf AFCC : {ref:>8,}, "
                        f"écart : {diff:+d}) {status}")
        else:
            lines.append(f"  {year} : {total:>8,}  (pas de référence AFCC)")
    lines.append("")
    lines.append("FICHIERS GÉNÉRÉS :")
    lines.append(f"  - {OUTPUT_DIR}/<annee>_raw.json (×{len(all_data)})")
    lines.append(f"  - {OUTPUT_DIR}/oficina_long.csv")
    lines.append(f"  - {OUTPUT_DIR}/rapport.txt (ce fichier)")
    
    content = "\n".join(lines)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print("\n" + content)
    return path


# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================
def main():
    print(f"Extraction Oficina del Peregrino : {list(YEARS)}")
    print(f"Sortie : {OUTPUT_DIR.absolute()}\n")
    
    all_data = {}
    errors = []
    
    for year in YEARS:
        print(f"→ {year}...", end=" ", flush=True)
        try:
            records = fetch_year(year)
            save_raw(records, year)
            all_data[year] = records
            print(f"OK ({len(records)} mois)")
        except requests.HTTPError as e:
            print(f"ERREUR HTTP {e.response.status_code}")
            errors.append((year, str(e)))
        except Exception as e:
            print(f"ERREUR {type(e).__name__}: {e}")
            errors.append((year, str(e)))
        
        if year != list(YEARS)[-1]:
            time.sleep(PAUSE_SECONDS)
    
    if not all_data:
        print("\n✗ Aucune donnée récupérée. Arrêt.")
        return
    
    print("\nConsolidation en format long...")
    columns, rows = consolidate_to_long(all_data)
    csv_path = save_long_csv(columns, rows)
    print(f"✓ {len(rows)} lignes écrites dans {csv_path}")
    
    generate_report(all_data, columns)
    
    if errors:
        print(f"\n⚠ {len(errors)} erreurs rencontrées :")
        for year, err in errors:
            print(f"  - {year}: {err}")


if __name__ == "__main__":
    main()


Extraction Oficina del Peregrino : [2018, 2019, 2020, 2021, 2022, 2023, 2024]
Sortie : c:\Users\cello\Desktop\FR_Santiago_caminos\oficina_raw

→ 2018... OK (12 mois)
→ 2019... OK (6 mois)
→ 2020... OK (0 mois)
→ 2021... OK (0 mois)
→ 2022... OK (0 mois)
→ 2023... OK (0 mois)
→ 2024... OK (0 mois)

Consolidation en format long...
✓ 18 lignes écrites dans oficina_raw\oficina_long.csv

RAPPORT D'EXTRACTION - OFICINA DEL PEREGRINO
Date extraction : 2026-04-17T09:57:41
Source          : https://oficinadelperegrino.com/wp-admin/admin-ajax.php
Période         : 2018-2024
Nombre de mois  : 18
Schéma détecté  : 17 colonnes

COLONNES DÉCOUVERTES :
  - anio
  - mes
  - mes_titulo
  - nb_registros
  - autonomias
  - camino
  - created_at
  - edades
  - medios
  - motivacion
  - nb_ultimo_anno_santo
  - paises
  - procedencia
  - profesiones
  - sexos
  - ultimo_anno_santo
  - updated_at

TOTAUX ANNUELS (contrôle cohérence vs AFCC) :
-----------------------------------------------------------------